# Import

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


## Load dataset

In [ ]:
macro_dataset = pd.read_csv('data/detailed_meals_macros.csv')

display(macro_dataset.head(5))
display(macro_dataset.info())

display(macro_dataset.describe().T)

## Basic Cleaning of  data
- Removing trailing spaces
- Drop rows with missing target variables
- Fill or drop missing values
-

## 5) Extensive cleanup and data source preparation

##### 5.1) Missing Values Ratio:
column-based deficiency ratio elimination


In [ ]:
# 5.1.1 – Calculate the missing percentages (ccy)
import pandas as pd
import numpy as np



missing_ratio_ccy = df.isna().mean().sort_values(ascending=False)
print("The 15 columns with the highest deficiency rate:\n", missing_ratio_ccy.head(15))

# Threshold: Let's exclude columns with a deficiency rate of 30% or higher (typically, a range of 20–40% is preferred in courses).
MISSING_THRESH = 0.30

drop_cols_missing_ccy = missing_ratio_ccy[missing_ratio_ccy > MISSING_THRESH].index.tolist()
print(f"\nShortage ratio > {MISSING_THRESH:.0%} Number of columns that exist and will be removed: {len(drop_cols_missing_ccy)}")
print(drop_cols_missing_ccy)

# 5.1.2 – remove columns
df_mv_filtered_ccy = df.drop(columns=drop_cols_missing_ccy)
print("\nFigure (before/after):", df.shape, "->", df_mv_filtered_ccy.shape)


##### 5.2) Low Variance Filter:
(elimination of low-variance features)

We proceed along two paths below:

-Numeric columns: VarianceThreshold (sklearn)

-Categorical columns: “near-zero variance” intuition (if the dominant category ratio is very high)

In [ ]:
# 5.2.0 – Seperation of spice
num_cols_ccy = df_mv_filtered_ccy.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_ccy = df_mv_filtered_ccy.select_dtypes(exclude=[np.number]).columns.tolist()
num_cols_ccy, cat_cols_ccy[:10]


##### 5.2.1) Low variance filter for numerical columns

In [ ]:
# 5.2.1 – 
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

num_imputer_ccy = SimpleImputer(strategy="median")
X_num_ccy = pd.DataFrame(
    num_imputer_ccy.fit_transform(df_mv_filtered_ccy[num_cols_ccy]),
    columns=num_cols_ccy,
    index=df_mv_filtered_ccy.index
)

# 
num_variances_ccy = X_num_ccy.var().sort_values()
print("The 10 columns with the lowest variance:\n", num_variances_ccy.head(10))

# Threshold selection:
# - If features are not standardized, 0.0 is a good starting point (discards completely constant or nearly constant variables)
# - Alternative: discard the bottom 5% (present in the comment line)
MIN_VAR_THRESH = 0.0
# MIN_VAR_THRESH = num_variances_ccy.quantile(0.05)

vt_ccy = VarianceThreshold(threshold=MIN_VAR_THRESH)
X_num_kept = vt_ccy.fit_transform(X_num_ccy)
kept_num_mask_ccy = vt_ccy.get_support()
kept_num_cols_ccy = list(X_num_ccy.columns[kept_num_mask_ccy])
dropped_num_cols_ccy = [c for c in X_num_ccy.columns if c not in kept_num_cols_ccy]

print(f"\nNumber of numerical columns discarded due to low variance: {len(dropped_num_cols_ccy)}")
print(dropped_num_cols_ccy)


##### 5.2.2) “Near-zero variance” filter for categorical columns

In [ ]:
# 5.2.2 –
from collections import Counter

def find_near_zero_var_cats_ccy(df_cat: pd.DataFrame,
                                dominant_thresh: float = 0.95,
                                unique_frac_thresh: float = 0.10):
    drop = []
    stats = {}
    n = len(df_cat)
    for col in df_cat.columns:
        vc = df_cat[col].astype("object")
        # To avoid thinking of missing values as a single category, let's exclude NaNs from the most frequent category count.
        counts = Counter([x for x in vc.dropna().tolist()])
        if n == 0:
            continue
        if counts:
            dom = counts.most_common(1)[0][1] / n
        else:
            dom = 1.0  # If all of them NaN, we count them near-zero in practice
        unique_frac = vc.nunique(dropna=True) / n
        stats[col] = {"dominant_ratio": dom, "unique_frac": unique_frac}
        if (dom >= dominant_thresh) or (unique_frac <= unique_frac_thresh):
            drop.append(col)
    return drop, pd.DataFrame(stats).T.sort_values("dominant_ratio", ascending=False)

drop_cat_cols_ccy, cat_stats_ccy = find_near_zero_var_cats_ccy(df_mv_filtered_ccy[cat_cols_ccy])
print("Candidate categorical columns that could have near-zero variance:\n", drop_cat_cols_ccy)
cat_stats_ccy.head(10)


##### 5.2.3) Create the final data frame

In [ ]:
# 5.2.3 – Remove all low-variance ones and summarize
df_lowvar_filtered_ccy = df_mv_filtered_ccy.drop(columns=dropped_num_cols_ccy + drop_cat_cols_ccy)

print("\nFigure (before/after):", df_mv_filtered_ccy.shape, "->", df_lowvar_filtered_ccy.shape)

feature_drop_report_ccy = {
    "dropped_missing_cols": drop_cols_missing_ccy,
    "dropped_low_variance_num": dropped_num_cols_ccy,
    "dropped_low_variance_cat": drop_cat_cols_ccy
}
feature_drop_report_ccy


##### Tukey's Ladder


In [ ]:
# 5.3.0 – 
import numpy as np, pandas as pd

try:
    base_df_ccy = df_lowvar_filtered_ccy.copy()
except NameError:
    base_df_ccy = df.copy()

# 5.3.1 – Numerical columns and initial skewness
num_cols_ccy = base_df_ccy.select_dtypes(include=[np.number]).columns.tolist()
skew_before_ccy = base_df_ccy[num_cols_ccy].skew().sort_values(ascending=False)
print("The 10 most skewed numerical columns (first):\n", skew_before_ccy.head(10))

# Candidate columns: |skew| > 1 (you can set it to 0.8/1.5, etc. if you want)
SKEW_THRESH = 1.0
skewed_cols_ccy = skew_before_ccy[skew_before_ccy.abs() > SKEW_THRESH].index.tolist()
print("\nNumber of columns to be applied:", len(skewed_cols_ccy))
print(skewed_cols_ccy)


In [ ]:
# 5.3.2 – Helpers: positive structure + transformation + selecting the best lambda
def _make_positive_ccy(s: pd.Series, eps: float = 1e-9):
    s = s.astype(float)
    minv = s.min()
    offset = (1 - minv) if minv <= 0 else 0.0
    return s + offset + eps, offset

def _tukey_apply_ccy(s_pos: pd.Series, lam: float):
    if lam == 0:
        return np.log(s_pos)           # log
    else:
        return np.power(s_pos, lam)    # x^λ (negative λ -> 1/x^{|λ|})

def tukey_best_lambda_ccy(s: pd.Series, lambdas=None):
    if lambdas is None:
        lambdas = [-2, -1, -0.5, 0, 0.5, 1, 2]
    s_pos, offset = _make_positive_ccy(s)
    cand = {}
    for lam in lambdas:
        tr = pd.Series(_tukey_apply_ccy(s_pos, lam), index=s.index)
        cand[lam] = tr
    # |skew| the smallest lambda
    best_lam = min(cand, key=lambda L: abs(cand[L].skew()))
    return cand[best_lam], best_lam, offset


In [ ]:
# 5.3.3 – Apply and report (ccy)
df_tukey_ccy = base_df_ccy.copy()
tukey_meta_ccy = {}

for col in skewed_cols_ccy:
    transformed, lam, offset = tukey_best_lambda_ccy(df_tukey_ccy[col])
    df_tukey_ccy[col] = transformed
    tukey_meta_ccy[col] = {"lambda": lam, "offset_added": offset}

# 5.3.4 – Before/After distortion comparison
skew_after_ccy = df_tukey_ccy[skewed_cols_ccy].skew().sort_values(ascending=False)

tukey_report_ccy = pd.DataFrame({
    "skew_before": skew_before_ccy.loc[skewed_cols_ccy],
    "skew_after":  skew_after_ccy,
    "lambda":      [tukey_meta_ccy[c]["lambda"] for c in skewed_cols_ccy],
    "offset":      [tukey_meta_ccy[c]["offset_added"] for c in skewed_cols_ccy],
}).sort_values("skew_before", ascending=False)

print("\nTukey Ladder report (in order of importance):")
display(tukey_report_ccy.head(20))

print("\nFigure (before/after):", base_df_ccy.shape, "->", df_tukey_ccy.shape)
